# Pet Waste Model Training File

This file uses ultralytics and python to train a YOLO26 object detection model to identify pet waste. Our dataset is imported from Roboflow where it has already been properly annotated and split into training and validation sets.

Additionally, this notebook is ran in Google Colab in order to utilize server GPUS that can complete this training much more efficiently and quickly than normal consumer deskptop or laptop GPUs

### Install Ultralytics and RoboFlow Library

In [1]:
!pip install -q roboflow --upgrade
!pip install -q ultralytics --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.3 MB/s eta 0:00:00


### Check and confirm system GPU and RAM

In [2]:
# Check GPU
gpu_info = !nvidia-smi # This confirms an NVIDIA GPU is present, otherwise CUDA training will not work
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

# Check RAM
import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Sun Apr 26 22:58:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### Configuration
(Edit this cell only to change settings and configurations like roboflow project, yolo model, and training parameters)

In [3]:
# === Roboflow ===
ROBOFLOW_API_KEY = "eopByftdkdG3rvyTqNMU"  # or set via environment variable and read it below
ROBOFLOW_WORKSPACE = "teamdoggiedoo"
ROBOFLOW_PROJECT = "apwcr_dataset2" # Project-ID
ROBOFLOW_VERSION = 2 # This is the project version
ROBOFLOW_FORMAT = "yolo26"

# If you prefer environment variables:
# import os
# ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY", ROBOFLOW_API_KEY)

# === Training ===
BASE_MODEL = "yolo26n.pt"     # starting checkpoint (e.g., yolo26n.pt, yolo26s.pt, etc.)
IMG_SIZE = 640               # image size used during training/inference (Ultralytics resizes)
EPOCHS = 75
BATCH = 32                   # -1 lets Ultralytics auto-select, or set an int (e.g., 16)
DEVICE = 0                   # 0 = first GPU in Colab; "cpu" for CPU-only
patience_num = 20

# Where to save runs in Colab
ULTRALYTICS_PROJECT = "runs/detect"
RUN_NAME = "petwaste_yolo26n_V2"

# === Inference / Testing ===
CONF_THRES = 0.25

### Import Roboflow and Download Dataset

In [4]:
from roboflow import Roboflow
from pathlib import Path


rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)

# Download dataset in YOLOv8 format (compatible with YOLO11)
dataset = version.download(ROBOFLOW_FORMAT)

DATA_YAML = Path(dataset.location) / "data.yaml"
print("Dataset downloaded to:", dataset.location)
print("data.yaml:", DATA_YAML)


loading Roboflow workspace...
loading Roboflow project...
Exporting format yolo26 in progress : 94.0%
Version export complete for yolo26 format



Extracting Dataset Version Zip to apwcr_dataset2-2 in yolo26:: 100%|██████████| 2049/2049 [00:01<00:00, 1291.20it/s]


Dataset downloaded to: /content/apwcr_dataset2-2
data.yaml: /content/apwcr_dataset2-2/data.yaml


### Check Dataset Import

*NOTE: Ultralytics applies common augmentations on the fly during training by default

In [5]:
# Optional: Inspect dataset structure
from pathlib import Path

root = Path(dataset.location)
for split in ["train", "valid", "test"]:
    img_dir = root / split / "images"
    lbl_dir = root / split / "labels"
    print(f"{split:5s} images:", len(list(img_dir.glob('*'))) if img_dir.exists() else 0,
          "| labels:", len(list(lbl_dir.glob('*'))) if lbl_dir.exists() else 0)

#Check Null Images imported correctly
empty = [p for p in (root/"train/labels").glob("*.txt") if p.stat().st_size == 0]
print("Empty label files (null images):", len(empty))

# Note: Ultralytics applies common augmentations on the fly during training by default.

train images: 725 | labels: 725
valid images: 199 | labels: 199
test  images: 98 | labels: 98
Empty label files (null images): 101


### Train YOLO Model on Training Dataset

In [6]:
from ultralytics import YOLO
from pathlib import Path

model = YOLO(BASE_MODEL)

train_results = model.train(
    task="detect",
    mode="train",
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    project=ULTRALYTICS_PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    patience = patience_num
)

# Best weights path (what you'll usually use for val/predict)
RUN_DIR = Path(train_results.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"

print("Run dir:", RUN_DIR)
print("Best weights:", BEST_WEIGHTS, "| exists:", BEST_WEIGHTS.exists())
print("Last weights:", LAST_WEIGHTS, "| exists:", LAST_WEIGHTS.exists())


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/apwcr_dataset2-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=75, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.

### Validate Trained Model on Validation Dataset

In [7]:
from ultralytics import YOLO

assert BEST_WEIGHTS.exists(), f"Best weights not found at {BEST_WEIGHTS}"

best_model = YOLO(str(BEST_WEIGHTS))
val_results = best_model.val(
    data=str(DATA_YAML),
    split="val",          # uses the 'valid' folder from Roboflow export
    imgsz=IMG_SIZE,
    device=DEVICE,
)

print(val_results)


Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1807.4±965.4 MB/s, size: 1044.5 KB)
val: Scanning /content/apwcr_dataset2-2/valid/labels.cache... 199 images, 18 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 199/199 64.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 4.7it/s 2.7s
                   all        199        258      0.926      0.821      0.899      0.652
Speed: 1.1ms preprocess, 3.7ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /content/runs/detect/val
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d74edabb980>
curves: ['Precision-Recall(B)', 

### Test Trained and Validated Model on Test Dataset

*This is Optional

In [ ]:
  from ultralytics import YOLO
from pathlib import Path

best_model = YOLO(str(BEST_WEIGHTS))

test_images_dir = Path(dataset.location) / "test" / "images"
assert test_images_dir.exists(), f"Test images dir not found: {test_images_dir}"

pred_results = best_model.predict(
    source=str(test_images_dir),
    conf=CONF_THRES,
    imgsz=IMG_SIZE,
    device=DEVICE,
    save=True,            # saves annotated images to the run folder
)

print(f"Predicted on {len(pred_results)} images")



image 1/83 /content/apwcr_dataset-1/test/images/74656a86-ad14-4e46-bb0f-02e238929805-1-_JPG.rf.8120142de153d29cd57098c10d2091d1.jpg: 640x480 1 pet-waste, 19.2ms
image 2/83 /content/apwcr_dataset-1/test/images/IMG_0044_jpg.rf.972b3bc4b15d24d733b6b2535c31f292.jpg: 640x480 1 pet-waste, 4.2ms
image 3/83 /content/apwcr_dataset-1/test/images/IMG_0048_jpg.rf.1cccb053821c54fb535518960183045d.jpg: 640x480 2 pet-wastes, 4.2ms
image 4/83 /content/apwcr_dataset-1/test/images/IMG_8072_jpg.rf.3bd73d1cf5700e2069afd15498724357.jpg: 480x640 (no detections), 20.0ms
image 5/83 /content/apwcr_dataset-1/test/images/IMG_8075_jpg.rf.34574f201df9141669b39114af9af89f.jpg: 480x640 3 pet-wastes, 4.3ms
image 6/83 /content/apwcr_dataset-1/test/images/IMG_8283_jpg.rf.a62d611d1857c295d736f6a63227a356.jpg: 640x480 4 pet-wastes, 4.5ms
image 7/83 /content/apwcr_dataset-1/test/images/IMG_8286_jpg.rf.dd1f1098d077b2e780c29bfbe3058d0e.jpg: 640x480 4 pet-wastes, 4.1ms
image 8/83 /content/apwcr_dataset-1/test/images/IMG_830